[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanjaynagi/malaria-software-training/blob/main/day-5/5-2-good-practice.ipynb)

# 5.2 Good practice in bioinformatics

**Theme:** Tools & technology

This morning you presented your gene dossiers. Each one draws on two workflows, four Python
packages and data from ENA, SRA, Zenodo and MalariaGEN. Could someone else rebuild your figures
from your repository? Could you, in a year?

On Day 1 (Module 1.2) you set up a project layout, pinned environments and a git repository.
This module looks back at how those habits worked during the week, and then covers what comes
next: using workflow outputs as a record of what you did, sharing data and code so others can
reuse them, licensing, and writing a README. It ends with a checklist to use on your own projects.

## Learning objectives

At the end of this module you will be able to:

- Audit a project against a reproducibility checklist and fix the gaps.
- Use workflow outputs (configs, logs, environment files, reports) as provenance, and write a Methods paragraph from them.
- Choose where to share raw reads, processed data and code, and what metadata to provide.
- Choose licences for code, text and data, and write a useful README.

## Setup

Only standard Python and pandas are used. Files are written to `course-work/` next to this notebook.

In [1]:
import json
import platform
import subprocess
import xml.etree.ElementTree as ET
from datetime import date
from importlib import metadata
from pathlib import Path
from urllib.request import urlopen

import pandas as pd

WORKDIR = Path("course-work").resolve()
WORKDIR.mkdir(exist_ok=True)

## Looking back at the week

| Habit (Module 1.2) | Where you used it |
|---|---|
| **Project layout** | Your `my-dossier/` folder. RNA-Seq-Pop and AmpSeeker use the standard Snakemake layout: `config/`, `workflow/`, `resources/`, `results/` |
| **Read-only raw data** | The Busia and Siaya reads were downloaded from ENA/SRA and never edited. Every result was generated from them by code |
| **Pinned environments** | The course's own `pixi.toml` and `pixi.lock` (and your practice pixi project in Module 1.2); pinned `%pip install` lines at the top of every Colab notebook; Snakemake installed with pixi in Codespaces, which then built a conda environment for each RNA-Seq-Pop and AmpSeeker rule with `--use-conda` |
| **Version control** | You committed to your fork every day |

Take a minute to look at your own history. In your codespace terminal:

```bash
git log --oneline -- my-dossier          # one line per commit
git log --stat -- my-dossier | head -40  # which files changed in each commit
```

If there are gaps (a day with no commits, or a commit called `stuff`), that is normal. The point
is that the history exists, and it gets easier with practice.

A useful question to ask about any figure in your dossier: **could I regenerate it from the
repository alone?** If the answer depends on a file on your laptop, or a cell you ran once and
then deleted, fix that first.

## Workflow outputs as provenance

**Provenance** is the record of how a result was produced: from which data, by which code, with
which software and settings. A workflow manager records most of this for you, as long as you
keep it.

| What Snakemake leaves behind | What it records | Keep it? |
|---|---|---|
| `config/config.yaml`, `config/samples.tsv` | Samples, contrasts, parameters | Commit to git |
| The workflow version (git tag or commit, e.g. an RNA-Seq-Pop release) | Exactly which code ran | Record it in your README and Methods |
| `workflow/envs/*.yaml` | Software versions for each step | Part of the workflow repo |
| `logs/` and `benchmarks/` | Each job's messages, run time and memory | Keep with the results |
| `.snakemake/log/*.log` | The full log of each run | Keep the log of the final run |
| `snakemake --report report.html` | Results, run times, code and software, in one HTML file | Share with results |
| Results books (RNA-Seq-Pop, AmpSeeker) | Figures and tables with the code that made them | Share (e.g. GitHub Pages) |

### Writing Methods from provenance

A Methods section for a workflow-based analysis should let a reader rerun it. It should include:

- **Data:** accession numbers (e.g. ENA PRJNA748581), and any subsetting.
- **Workflow:** name, version (release or commit) and a link.
- **Configuration:** the config file (in a supplement or repository), and key parameters and thresholds.
- **Reference data:** genome and annotation versions (e.g. AgamP4 with a named VectorBase release).
- **Key tools and versions:** read from the workflow's environment files.
- **Statistics:** tests, multiple-testing correction, thresholds.

For example, for the course's worked example gene, CYP6AA1 (AGAP002862), with placeholders to
fill in from your own run:

> Reads were analysed with RNA-Seq-Pop *vX.Y.Z* (commit `abc1234`; configuration in
> Supplementary File 1). Reads were pseudo-aligned to the AgamP4 transcriptome
> (VectorBase release *NN*) with kallisto *vX*, and differential expression was tested with
> DESeq2 *vX* for the BusRes vs BusSus and BusSus vs Kisumu contrasts, using a
> Benjamini–Hochberg adjusted *p* < 0.05. Downstream analyses of CYP6AA1 (AGAP002862) were run
> in notebooks using the environment in `pixi.lock` (commit `def5678`).

### Practical: record your own environment

If you work locally with pixi, `pixi.lock` already records every package version, and the git
commit records which `pixi.lock` you used. Colab notebooks keep no such record unless you ask.
This cell writes a small provenance file for your dossier: the date, Python version, versions of
the course packages that are installed, and the git commit if you are in a repository.

In [2]:
def provenance(packages, repo="."):
    """Return a dict describing the current software environment."""
    record = {
        "date": date.today().isoformat(),
        "python": platform.python_version(),
        "platform": platform.platform(terse=True),
        "packages": {},
    }
    for pkg in packages:
        try:
            record["packages"][pkg] = metadata.version(pkg)
        except metadata.PackageNotFoundError:
            record["packages"][pkg] = "not installed"
    git = subprocess.run(["git", "-C", repo, "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True)
    record["git_commit"] = git.stdout.strip() if git.returncode == 0 else None
    return record


record = provenance(["pandas", "malariagen_data", "anoexpress", "anoprimer", "snakemake"])
Path(WORKDIR / "provenance.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

{
  "date": "2026-09-26",
  "python": "3.12.14",
  "platform": "macOS-26.5.2",
  "packages": {
    "pandas": "2.2.3",
    "malariagen_data": "not installed",
    "anoexpress": "not installed",
    "anoprimer": "not installed",
    "snakemake": "9.27.0"
  },
  "git_commit": null
}


Run this at the end of each analysis notebook and commit the output alongside your figures.

## Sharing data

Different parts of a project go to different places:

| What | Where | Why there |
|---|---|---|
| Raw sequence reads | **ENA** (EMBL-EBI), **SRA** (NCBI) or DDBJ. These three form the INSDC and mirror each other | Permanent, free, expected by journals; accession numbers let anyone download the reads |
| Processed data (count tables, VCFs, results) and code snapshots | **Zenodo** (or Figshare, Dryad) | A citable **DOI** for each version; large files allowed; GitHub releases can be archived automatically |
| Code under development | **GitHub** | History, issues, collaboration. Not an archive on its own: repositories can be deleted, so archive releases on Zenodo |
| Genotypes for surveillance | Community resources, e.g. MalariaGEN data releases | Combined with other studies |

INSDC archives organise submissions as a **study** (BioProject, e.g. PRJNA748581), **samples**
(BioSample, e.g. SAMN...), **experiments** (library details) and **runs** (the FASTQ files,
e.g. SRR...). The course mini datasets are on Zenodo so that every notebook can cite a fixed,
versioned copy.

The aim is data that are **FAIR**: findable, accessible, interoperable and reusable [1].

### Metadata

Reads without metadata are almost useless. Archives provide sample checklists based on the
MIxS standards [2]. For mosquito resistance studies, the fields that matter most are:

- species, and how it was identified (morphology, PCR, AIMs)
- collection location (country, place, latitude and longitude) and date
- collection method and life stage, sex and age
- for colonies: origin, year established, and selection history
- **phenotype**: insecticide, dose, assay type (e.g. WHO tube, net exposure) and outcome (dead/alive)
- tissue, library preparation and sequencing platform

### Practical: how good is the Busia metadata?

Let's look at what ENA holds for the Busia RNA-seq samples.

In [3]:
url = ("https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA748581"
       "&result=read_run&format=tsv&fields="
       "run_accession,sample_accession,sample_alias,sample_title,country,collection_date,"
       "instrument_model,library_strategy")
busia = pd.read_csv(url, sep="\t").sort_values("sample_alias")
busia

,run_accession,sample_accession,sample_alias,sample_title,country,collection_date,instrument_model,library_strategy
3,SRR15206227,SAMN20339596,BusRes1,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
13,SRR15206226,SAMN20339597,BusRes2,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
6,SRR15206219,SAMN20339598,BusRes3,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
9,SRR15206218,SAMN20339599,BusRes4,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
15,SRR15206217,SAMN20339600,BusRes5,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
8,SRR15206216,SAMN20339601,BusRes6,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
0,SRR15206215,SAMN20339602,BusSus1,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
5,SRR15206214,SAMN20339603,BusSus2,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
14,SRR15206213,SAMN20339604,BusSus3,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq
4,SRR15206212,SAMN20339605,BusSus4,Invertebrate sample from Anopheles gambiae,Uganda: Busia,2020-12,Illumina NovaSeq 6000,RNA-Seq


Now the full set of attributes for one sample, from the ENA browser API:

In [4]:
sample = busia.sample_accession.iloc[0]
xml = urlopen(f"https://www.ebi.ac.uk/ena/browser/api/xml/{sample}").read()
attributes = {
    a.findtext("TAG"): a.findtext("VALUE")
    for a in ET.fromstring(xml).iter("SAMPLE_ATTRIBUTE")
}
pd.Series(attributes, name=sample)

collection_date                        2020-12
isolate                                BusRes1
replicate                                    1
INSDC secondary accession           SRS9550903
NCBI submission package       Invertebrate.1.0
organism                     Anopheles gambiae
sex                                     female
host                                        NA
geo_loc_name                     Uganda: Busia
tissue                                   Whole
age                                   3-4 days
BioSampleModel                    Invertebrate
ENA-FIRST-PUBLIC                    2022-08-31
ENA-LAST-UPDATE                     2022-08-31
Name: SAMN20339596, dtype: object

## Licensing

Work with **no licence** is, legally, "all rights reserved": others cannot safely reuse it, even
if it is public on GitHub. Add a `LICENSE` file to every repository.

| Type of work | Common choices | Notes |
|---|---|---|
| Code | **MIT** or **BSD-3-Clause** (permissive); **GPL-3.0** (copyleft: derived code must stay open) | Permissive licences are the most common for research software |
| Text, figures, teaching material | **CC BY 4.0**; **CC BY-SA 4.0** (derived works must use the same licence) | This course is CC BY-SA 4.0, like the PAMCA course it builds on |
| Data | **CC0** (no restrictions) or **CC BY 4.0** (attribution) | Check any data-sharing agreements first |

https://choosealicense.com explains the code licences in plain language.

Data about mosquitoes collected in partner countries may also be covered by collaboration and
data-sharing agreements, national permits and benefit-sharing rules. Agree how data will be
shared, and how field teams will be credited, **before** sampling, not at the point of
publication.

## Writing a good README

The README is the first (often the only) file people read. A good one answers:

1. **What is this?** One or two sentences, and the question it addresses.
2. **Where is the data?** Accessions or DOIs, and how to download them.
3. **How do I install the software?** The environment file and the command to create it (e.g. `pixi install`).
4. **How do I run it?** The exact commands, in order.
5. **What comes out?** Where the main results and figures appear.
6. **How is it organised?** A short description of the folders.
7. **How do I cite it, what is the licence, and who do I contact?**

### Practical: a README for your dossier

This writes a README template into `course-work/my-dossier/`. Copy it into the `my-dossier/`
folder of your fork, fill it in, and commit it.

In [5]:
dossier = WORKDIR / "my-dossier"
dossier.mkdir(exist_ok=True)

In [6]:
%%writefile course-work/my-dossier/README.md
# Gene dossier: <GENE NAME> (<AGAP ID>), e.g. CYP6AA1 (AGAP002862)

Evidence that <GENE> contributes to pyrethroid resistance in *Anopheles gambiae* around
Lake Victoria, from RNA-seq, meta-analysis, whole genomes, selection scans, assay design and
amplicon sequencing. Produced during the malaria vector software course.

## Data
| Data | Source |
|---|---|
| Busia RNA-seq | ENA PRJNA748581; RNA-Seq-Pop results (Zenodo DOI: <...>) |
| Expression meta-analysis | AnoExpress <version> |
| Whole genomes | MalariaGEN Ag3 via malariagen_data <version> |
| Selection signals | Malaria vector selection atlas (accessed <date>) |
| Siaya amplicons | SRA PRJNA1207724; AmpSeeker <version> |

## Reproducing
- **Locally:** install [pixi](https://pixi.sh), then `pixi install` (recreates the environment from
  `pixi.lock`) and `pixi run lab`. Run the notebooks in `notebooks/` in numbered order.
- **In Colab:** open each notebook in numbered order. Package versions are pinned in its first
  cell and recorded in `provenance.json`.

## Layout
- `notebooks/`: analysis notebooks, numbered in run order
- `results/`: tables and figures produced by the notebooks
- `dossier.md`: the dossier itself

## Licence and contact
Text and figures: CC BY 4.0. Code: MIT. Contact: <your name, email>.

Writing course-work/my-dossier/README.md


## A checklist for your own projects

Use this at the start of a project and again before you share it. It condenses the
recommendations in [3–5]; you do not need every item on day one.

**Organisation**
- [ ] One folder and one git repository per project, with a standard layout
- [ ] Relative paths only; file names without spaces; dates as `YYYY-MM-DD`

**Data**
- [ ] Raw data read-only, with source, date and checksum recorded
- [ ] Metadata in a plain-text table, with the fields a stranger would need
- [ ] Every result can be regenerated from raw data by code

**Software**
- [ ] An environment file committed with the code: `pixi.toml` + `pixi.lock` (recommended), or `environment.yml` / pinned pip
- [ ] Versions of key tools and the workflow recorded with the results

**Workflows**
- [ ] Multi-step or multi-sample analyses run through a workflow manager
- [ ] Config files committed; logs and reports kept with results

**Version control**
- [ ] Small, frequent commits with clear messages
- [ ] No large data or secrets in the repository (`.gitignore`)

**Sharing**
- [ ] Raw reads in ENA/SRA with good sample metadata
- [ ] Processed data and code releases on Zenodo with a DOI
- [ ] A README, a LICENSE and a citation
- [ ] Agreements with partners and data owners respected, and field teams credited

### Practical: an automatic audit

The function below checks a project folder for some of the items above. It cannot check
everything, but it catches the most common gaps.

In [7]:
def audit_project(path):
    """Check a project folder for common reproducibility gaps."""
    p = Path(path)
    env_files = ["pixi.toml", "environment.yml", "environment.yaml", "requirements.txt", "pyproject.toml"]
    lock_files = ["pixi.lock", "conda-lock.yml", "uv.lock"]
    checks = {
        "README": any(p.glob("README*")),
        "LICENSE": any(p.glob("LICENSE*")) or any(p.glob("LICENCE*")),
        "environment file": any((p / f).exists() for f in env_files),
        "lock file": any((p / f).exists() for f in lock_files),
        ".gitignore": (p / ".gitignore").exists(),
        "git repository": (p / ".git").exists(),
        "no files over 50 MB": not any(
            f.is_file() and f.stat().st_size > 50e6
            for f in p.rglob("*") if ".git" not in f.parts
        ),
        "no spaces in file names": not any(
            " " in f.name for f in p.rglob("*") if ".git" not in f.parts
        ),
    }
    return pd.DataFrame({"check": list(checks), "ok": list(checks.values())})


audit_project(dossier)

,check,ok
0,README,True
1,LICENSE,False
2,environment file,False
3,lock file,False
4,.gitignore,False
5,git repository,False
6,no files over 50 MB,True
7,no spaces in file names,True


## Exercises

### Exercise 1

Look at the Busia sample metadata above. The samples fall into three groups (Kisumu, Busia
unselected and Busia deltamethrin-selected). Where in the metadata is the group recorded? List
three other fields you would add, or make clearer, so that someone could reuse these data
without reading the paper.

<details><summary>Show answer</summary>

The group is only recorded in the sample **alias** (`Kis`, `BusSus`, `BusRes`), which is a
free-text label. There is no field for insecticide selection. Useful additions:

- a structured **treatment/phenotype** field (e.g. "deltamethrin-selected colony", "unselected
  parental colony", "susceptible reference strain"), and the selection regime (insecticide,
  dose, number of generations);
- **colony origin**: where and when the wild founders were collected, and when the colony was
  established. The `collection_date` (2020-12) is presumably when these colony mosquitoes were
  sampled for RNA, not when the population was collected from the field;
- for Kisumu, that it is a long-established **laboratory strain**: the `geo_loc_name`
  "Kenya: Kisumu" could be misread as a recent field collection;
- a more informative `sample_title` than the default "Invertebrate sample from *Anopheles gambiae*".

The sex, age and tissue fields are filled in, which is good.

</details>

### Exercise 2

Run `audit_project()` on the dossier folder above, then fix every failing check (you can
create files from Python or with `%%bash`). Rerun the audit. If you are in your codespace, run
it on your fork too: `audit_project("/workspaces/malaria-software-training")`.

<details><summary>Show answer</summary>

```python
(dossier / "LICENSE").write_text("MIT License\n\nCopyright (c) 2026 <your name>\n...\n")
(dossier / ".gitignore").write_text("data/raw/\n")
subprocess.run(["git", "init", "-q", str(dossier)])
```

For the environment and lock file, use pixi (install it first as in Module 1.2 if you are in Colab):

```bash
%%bash
cd course-work/my-dossier
pixi init .                       # adds .pixi/ to .gitignore
pixi add python=3.11 pandas=2.2.2
pixi add --pypi malariagen_data==15.9.0
```

Then rerun `audit_project(dossier)`.

Use the full licence text from https://choosealicense.com rather than a placeholder. In your
fork, the repository root already has a README, `pixi.toml` and `pixi.lock`, but `my-dossier/`
should still have its own README.

</details>

### Exercise 3

Write a Methods paragraph (4–6 sentences) for the Busia differential expression result for
CYP6AA1 (AGAP002862) or one of your own genes. Which values can you fill in from the workflow outputs, and which do you still
need to look up?

<details><summary>Show answer</summary>

A good paragraph names the data (PRJNA748581, the three groups and replicate numbers), the
workflow and its version (RNA-Seq-Pop release or commit), the reference (AgamP4 and the
VectorBase release), the quantification and DE tools with versions (kallisto, DESeq2), the
contrast (e.g. BusRes vs BusSus) and the significance threshold.

Most of these come straight from the workflow: the config file (samples, contrasts,
thresholds), the environment files (tool versions) and the git history (workflow version).
Things usually missing, which you have to look up, are details of the biology: how and when
the colony was established, the selection regime, and mosquito age and rearing conditions.

</details>

## Quiz

Check your understanding. The quiz runs in Colab and in the course book.

In [ ]:
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
display_quiz("https://raw.githubusercontent.com/sanjaynagi/malaria-software-training/main/quizzes/5-2-good-practice.json")

## Summary

- The habits from Day 1 (layout, environments, git) are what make your dossier reproducible.
  Check that every figure can be regenerated from your repository.
- Keep the environment with the project: `pixi.toml` and `pixi.lock` locally, pinned `%pip`
  lines in Colab. Workflows add per-rule conda environments on top.
- Workflow outputs (configs, environment files, logs, reports, the workflow version) are a
  ready-made provenance record. Keep them, and use them to write your Methods.
- Share raw reads in ENA/SRA with good metadata, and processed data and code on Zenodo with a DOI.
  Aim for FAIR data.
- Always add a licence and a README.

## Well done!

You have reached the end of the course. The rest of today is flexi time: work on your own
projects or data, or extend your dossier. Keep committing to your fork, and delete your
codespace when you have finished with it to free your storage allowance.

## References

1. Wilkinson MD *et al.* (2016). The FAIR Guiding Principles for scientific data management and stewardship. *Scientific Data*. https://doi.org/10.1038/sdata.2016.18
2. Yilmaz P *et al.* (2011). Minimum information about a marker gene sequence (MIMARKS) and minimum information about any (x) sequence (MIxS) specifications. *Nature Biotechnology*. https://doi.org/10.1038/nbt.1823
3. Wilson G *et al.* (2017). Good enough practices in scientific computing. *PLoS Computational Biology*. https://doi.org/10.1371/journal.pcbi.1005510
4. Sandve GK *et al.* (2013). Ten simple rules for reproducible computational research. *PLoS Computational Biology*. https://doi.org/10.1371/journal.pcbi.1003285
5. Buffalo V (2015). *Bioinformatics Data Skills: Reproducible and Robust Research with Open Source Tools*. O'Reilly Media. ISBN 978-1-4493-6737-4.
6. Perez-Riverol Y *et al.* (2016). Ten simple rules for taking advantage of Git and GitHub. *PLoS Computational Biology*. https://doi.org/10.1371/journal.pcbi.1004947